## Google Places API: Restaurant Data

This script uses the Google Places Text Search API to retrieve restaurant data for a given query (e.g., New York). It requests multiple pages of results, extracts key fields (name, location, type), removes duplicate entries based on `place_id`, and saves the final dataset as a CSV file.

In [ ]:
import time
import requests
import pandas as pd

# key and url
API_KEY = "put your API key here"
URL = "https://places.googleapis.com/v1/places:searchText"

# inputs
TEXT_QUERY = "Restaurants in New York"
INCLUDED_TYPE = "restaurant"
PAGE_SIZE = 20
MAX_PAGES = 5
OUTPUT_FILE = TEXT_QUERY + ".csv"

# headers
HEADERS = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": ",".join([
        "places.id",
        "places.displayName",
        "places.primaryType",
        "places.formattedAddress",
        "places.location",
        "nextPageToken"
    ])
}
#
def fetch_page(page_token=None):
    payload = {
        "textQuery": TEXT_QUERY,
        "includedType": INCLUDED_TYPE,
        "pageSize": PAGE_SIZE
    }

    if page_token:
        payload["pageToken"] = page_token

    response = requests.post(URL, headers=HEADERS, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()

#
def place_to_row(place):
    return {
        "place_id": place.get("id"),
        "name": place.get("displayName", {}).get("text"),
        "primary_type": place.get("primaryType"),
        "formatted_address": place.get("formattedAddress"),
        "latitude": place.get("location", {}).get("latitude"),
        "longitude": place.get("location", {}).get("longitude")
    }

#
def main():
    if not API_KEY:
        raise ValueError("API_KEY is empty.")

    places = []
    page_token = None

    for page in range(MAX_PAGES):
        data = fetch_page(page_token)
        places.extend(data.get("places", []))

        page_token = data.get("nextPageToken")
        print(f"Page {page + 1}: {len(data.get('places', []))} rows")

        if not page_token:
            break

        time.sleep(2)

    df = pd.DataFrame([place_to_row(place) for place in places])
    df = df.drop_duplicates(subset="place_id").reset_index(drop=True)
    df.to_csv(OUTPUT_FILE, index=False)

    print(df.info())
    print(f"Saved {len(df)} unique rows to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()